# tensor-zeros-init — ex6: histogram via scatter into a zeros buffer

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-zeros-init`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch zero-init — quick refresher

**The allocate-then-scatter pattern.** Pre-allocate a buffer of the right `(shape, dtype, device)` with `t.zeros(...)`, then write per-element results into it via indexed assignment or `index_add_` / `scatter_add_`. This is faster and clearer than `list.append` + `t.stack`, and it's the canonical move for histograms, confusion matrices, depth buffers, and any per-ray accumulator.

**Dtype matters.** Default is `float32`. Index buffers MUST be `t.long`. Counters should be `t.long` (or `t.int64`). Use `t.zeros_like(x)` when you want a fresh buffer that mirrors `x.shape + x.dtype + x.device` exactly.

### Exercise 6 — histogram via scatter into a zeros buffer

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Allocate a `(n_bins,)` integer zero buffer and accumulate counts into it via `index_add_`; then plot the histogram with matplotlib.
> Keywords: histogram, scatter-add, visualization, bincount
> ```

**KCs targeted:** `zeros-1d-shape`, `zeros-dtype-control`, `zeros-allocate-then-fill`

Implement `ex6_histogram(samples, n_bins)`. Build a 1-D histogram the manual way:

1. Allocate a `(n_bins,)` zero counter buffer with `dtype=t.long`.
2. For each value in `samples` (which are already integer bin indices in `[0, n_bins)`), increment the matching counter by 1. Use `counts.index_add_(0, samples, t.ones_like(samples))` so the scatter happens in one shot.
3. Return the counts tensor.

Inputs:
- `samples`: 1-D `t.long` tensor, values in `[0, n_bins)`.
- `n_bins`: int.

Output: `(n_bins,)` `t.long` tensor whose sum equals `len(samples)`.

After the test passes, the visualization cell below the solution draws the histogram as a matplotlib bar chart so you can see the distribution your scatter produced.

In [ ]:
def ex6_histogram(samples: Tensor, n_bins: int) -> Tensor:
    """Allocate (n_bins,) long zeros, scatter-add 1 per sample, return counts."""
    raise NotImplementedError()


def _test_ex6():
    samples = t.tensor([0, 0, 0, 2, 2, 5, 5, 5, 5, 9], dtype=t.long)
    counts = ex6_histogram(samples, n_bins=10)
    assert counts.shape == (10,), f'expected (10,), got {tuple(counts.shape)}'
    assert counts.dtype == t.long, f'expected dtype long, got {counts.dtype}'
    expected = t.tensor([3, 0, 2, 0, 0, 4, 0, 0, 0, 1], dtype=t.long)
    assert t.equal(counts, expected), f'value mismatch:\n{counts}\nvs\n{expected}'
    assert counts.sum().item() == len(samples), 'sum of counts must equal len(samples)'
    # Edge case — empty samples must yield an all-zero buffer (not error).
    empty_counts = ex6_histogram(t.zeros(0, dtype=t.long), n_bins=4)
    assert empty_counts.shape == (4,) and empty_counts.sum().item() == 0, 'empty samples must yield all-zero counts'

    # --- Visualization (only runs if the assertions above passed) ---
    rng = t.Generator().manual_seed(42)
    big_samples = t.randint(0, 20, (500,), generator=rng)
    big_counts = ex6_histogram(big_samples, n_bins=20)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(range(20), big_counts.tolist(), color='steelblue', edgecolor='black')
    ax.set_xlabel('bin index')
    ax.set_ylabel('count')
    ax.set_title(f'ex6 histogram — 500 samples into 20 bins (sum={big_counts.sum().item()})')
    ax.set_xticks(range(20))
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_histogram(samples: Tensor, n_bins: int) -> Tensor:
    counts = t.zeros(n_bins, dtype=t.long)
    counts.index_add_(0, samples, t.ones_like(samples))
    return counts
```

**Why `index_add_` and not a Python `for` loop?** `index_add_` runs the scatter as a single fused op — no Python overhead per element. For 500 samples it's noticeable; for 5M samples (one frame of a Ray Tracing accumulator) it's the difference between 50ms and 30s.

**Why `dtype=t.long` for the counter?** Counts are integers. A `float32` counter quietly loses precision once counts exceed ~16M (the float32 mantissa runs out). `t.long` (int64) handles up to 9.2e18 counts.

**Alternative one-liner.** `t.bincount(samples, minlength=n_bins)` does the same thing and is even more idiomatic — but this exercise drills the allocate-then-scatter pattern explicitly so you see the machinery `bincount` hides.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()